In [ ]:
from termcolor import colored
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from matplotlib import pyplot as plt
import matplotlib.image as mpimg
import matplotlib.cm as cm
from osgeo import gdal
import os

import rasterio
from rasterio.mask import mask
from rasterio.plot import show

#import funciones as fn

import sys
sys.path.append('../')

import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Defino variables que voy a estar usando:
def plote_porcent(banda_del_arreglo, banda='banda', p=0, nodata=None, figsize=(12,6)):
    band = np.asarray(banda_del_arreglo).astype(float)

    # — limpiar nodata —
    if nodata is not None:
        band[band == nodata] = np.nan

    # — estadísticas ignorando NaN —
    min_b, max_b = np.nanmin(band), np.nanmax(band)
    p_b_min = np.nanpercentile(band, p)
    p_b_max = np.nanpercentile(band, 100 - p)

    plt.figure(figsize=figsize)
    plt.hist(band[~np.isnan(band)].ravel(), bins=100, color='steelblue', alpha=0.7)

    # Línea y etiqueta del percentil bajo
    plt.axvline(p_b_min, color='red', linestyle='--', label=f'{p} %')
    plt.text(p_b_min, plt.ylim()[1]*0.9, f'{p_b_min:.2f}', color='red', ha='right', va='top')

    # Línea y etiqueta del percentil alto
    plt.axvline(p_b_max, color='black', linestyle='--', label=f'{100-p} %')
    plt.text(p_b_max, plt.ylim()[1]*0.9, f'{p_b_max:.2f}', color='black', ha='left', va='top')

    plt.title(f'Histograma {banda}')
    plt.xlabel('Valor')
    plt.ylabel('Frecuencia')
    plt.legend()
    plt.show()
#La redefino para limitar el ploteo hasta percentiles 0.01 por si la data esta muy variada:
def plote_porcent(banda_del_arreglo, banda='banda', p=0, nodata=None, figsize=(12,6)):
    band = np.asarray(banda_del_arreglo, dtype=float)
    # — limpiar nodata —
    if nodata is not None:
        band[band == nodata] = np.nan
    band_val = band[~np.isnan(band)].ravel()
    # — percentiles para recortar ejes —
    p01  = np.nanpercentile(band_val, 0.01)
    p999 = np.nanpercentile(band_val, 99.99)
    # — percentiles solicitados por el usuario —
    p_b_min = np.nanpercentile(band_val, p)
    p_b_max = np.nanpercentile(band_val, 100 - p)
    plt.figure(figsize=figsize)
    plt.hist(band_val, bins=100, color='steelblue', alpha=0.7, range=(p01, p999))
    # Líneas y etiquetas del percentil bajo/alto elegido
    plt.axvline(p_b_min, color='red', linestyle='--', label=f'{p} %')
    plt.text(p_b_min, plt.ylim()[1]*0.9, f'{p_b_min:.2f}', color='red', ha='right', va='top')
    plt.axvline(p_b_max, color='black', linestyle='--', label=f'{100-p} %')
    plt.text(p_b_max, plt.ylim()[1]*0.9, f'{p_b_max:.2f}', color='black', ha='left', va='top')
    # Limitar eje x a [0.01 %, 99.99 %]
    plt.xlim(p01, p999)

    plt.title(f'Histograma {banda}')
    plt.xlabel('Valor')
    plt.ylabel('Frecuencia')
    plt.legend()
    plt.show()

def porcentajes(banda_del_arreglo, p=0, nodata=None):
    band = np.asarray(banda_del_arreglo).astype(float)
    if nodata is not None:
        band[band == nodata] = np.nan

    min_b, max_b = np.nanmin(band), np.nanmax(band)
    p_b_min = np.nanpercentile(band, p)
    p_b_max = np.nanpercentile(band, 100 - p)
    print(f'Mínimo: {min_b}, Máximo: {max_b}')
    print(f'Percentil {p}%: {p_b_min}, Percentil {100-p}%: {p_b_max}')

def obtener_percentiles(banda_del_arreglo, p=0, nodata=None):
    band = np.asarray(banda_del_arreglo).astype(float)
    if nodata is not None:
        band[band == nodata] = np.nan

    low  = np.nanpercentile(band, p)
    high = np.nanpercentile(band, 100 - p)

    # Crear nombres válidos para variables (ejemplo multiplicando por 10 y convirtiendo a entero)
    p_str = str(int(p*100))
    p_inv_str = str(int((100 - p)*100))

    globals()[f'percentil_{p_str}'] = low
    globals()[f'percentil_{p_inv_str}'] = high

    return low, high

In [ ]:
# MODIFICAR direcciorios, segun cual elijas mover (o los 2) (*)

dir_DEM_o = 'C:/' # MODIFICAR

dir_DEM_proc = 'C:/DEM_procesando' # MODIFICAR

DEM_i = 'tu_DEM_fechaInicial' # MODIFICAR
DEM_f = 'tu_DEM_fechaFinal' # MODIFICAR

archivo_gl = f'{dir_DEM_proc}/shapes/tu_vector_limite_del_glaciar.shp' # MODIFICAR por tu carpeta

archivo_DEM_i = f'{dir_DEM_o}/{DEM_i}.tif'
archivo_DEM_f = f'{dir_DEM_o}/{DEM_f}.tif'
archivo_DEM_i_2 = f'{dir_DEM_proc}/{DEM_i}_movido.tif' # ó puede ser  = archivo_DEM_i # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_i_2b = f'{dir_DEM_proc}/{DEM_i}_movido_nan.tif' # ó puede ser  = archivo_DEM_i # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_f_2 = f'{dir_DEM_proc}/{DEM_f}_movido.tif' # ó puede ser  = archivo_DEM_f # Si no estaban desplazados entre ellos en la horizontal
archivo_DEM_f_2b = f'{dir_DEM_proc}/{DEM_f}_movido_nan.tif' # ó puede ser  = archivo_DEM_f # Si no estaban desplazados entre ellos en la horizontal

archivo_DEM_i_3 = f'{dir_DEM_proc}/{DEM_i}_igualadoH.tif' # ó puede ser  = archivo_DEM_i # ó puede ser  = archivo_DEM_i_2 # ó puede ser  = archivo_DEM_i_2b
archivo_DEM_f_3 = f'{dir_DEM_proc}/{DEM_f}_igualadoH.tif' # ó puede ser  = archivo_DEM_f # ó puede ser  = archivo_DEM_f_2 # ó puede ser  = archivo_DEM_f_2b

archivo_DEM_i_4 = f'{dir_DEM_proc}/{DEM_i}_corregidoH.tif' # ó puede ser  = archivo_DEM_i # ó puede ser  = archivo_DEM_i_2 # ó puede ser  = archivo_DEM_i_2b # ó puede ser  = archivo_DEM_i_3
archivo_DEM_f_4 = f'{dir_DEM_proc}/{DEM_f}_corregidoH.tif' # ó puede ser  = archivo_DEM_f # ó puede ser  = archivo_DEM_f_2 # ó puede ser  = archivo_DEM_f_2b # ó puede ser  = archivo_DEM_f_3

ruta_salida_dif = f'{dir_DEM_proc}/diferencia_f-i_gl.tif'

Una vez georeferenciados ambos DEM y corregidos verticalmente (al menos entre ellos, ver codigo 02) 

OJO, buscar las indicaciones en todo el código que digan: ***# MODIFICAR*** porque son las variables y datos que **tiene que cambiar el usuario del código**

# CORTAR
ambos DEMs por el limite del glaciar (guardado previamente en shapefile)

###  Yo uso:  
**archivo_DEM_i** que esta georreferenciado  
**archivo_DEM_f_3** que es el _f que corregío horizontalmente y verticalmente respecto de archivo_DEM_i, el cual estaba correcto de entrada en X e Y (usando los valores sin datos como nan en código 01)

*(Tené en cuenta que no necesito moverlos verticalmetne a una referencia porque determino la diferencia entre alturas en momento final y momento inicial y no me importa si estan flotando ambos a X metros de la altura real del terreno)*

In [ ]:
# DEM final

# Ruta del archivo raster y del shape
ruta_raster = archivo_DEM_f_3 # MODIFICAR en caso de que no necesitate mover horizontalmente o verticalmente
ruta_shape = archivo_gl

# Cargar el shapefile
shape = gpd.read_file(ruta_shape)

# Asegurarse de que el CRS del shapefile y el raster coinciden
with rasterio.open(ruta_raster) as src:
    if shape.crs != src.crs:
        shape = shape.to_crs(src.crs)
    
    geoms = shape.geometry.values
    geometry = [geom.__geo_interface__ for geom in geoms]
    
    # Recortar el raster usando la máscara
    out_image, out_transform = mask(src, geometry, crop=True, nodata=np.nan)
    
    # Copiar los metadatos del raster original
    out_meta = src.meta.copy()
    
    # Cambiar el tipo de datos a flotante si es necesario para soportar NaN
    out_meta.update(dtype=rasterio.float32, nodata=np.nan)

# Actualizar los metadatos para reflejar las nuevas dimensiones del raster recortado
out_meta.update({"driver": "GTiff",
                 "height": out_image.shape[1],
                 "width": out_image.shape[2],
                 "transform": out_transform})

# Ruta de salida del raster recortado
ruta_salida_f = ruta_raster.replace('.tif', '_gl_CaboLamb.tif')

# Escribir el raster recortado en el disco
with rasterio.open(ruta_salida_f, "w", **out_meta) as dest:
    dest.write(out_image.astype(rasterio.float32))

In [ ]:
# DEM inicial

# Ruta del archivo raster y del shape
ruta_raster = archivo_DEM_i # MODIFICAR en caso de que lo hayas movido
ruta_shape = archivo_gl

# Cargar el shapefile
shape = gpd.read_file(ruta_shape)

# Asegurarse de que el CRS del shapefile y el raster coinciden
with rasterio.open(ruta_raster) as src:
    if shape.crs != src.crs:
        shape = shape.to_crs(src.crs)
    
    geoms = shape.geometry.values
    geometry = [geom.__geo_interface__ for geom in geoms]
    
    # Recortar el raster usando la máscara
    out_image, out_transform = mask(src, geometry, crop=True, nodata=np.nan)
    
    # Copiar los metadatos del raster original
    out_meta = src.meta.copy()
    
    # Cambiar el tipo de datos a flotante si es necesario para soportar NaN
    out_meta.update(dtype=rasterio.float32, nodata=np.nan)

# Actualizar los metadatos para reflejar las nuevas dimensiones del raster recortado
out_meta.update({"driver": "GTiff",
                 "height": out_image.shape[1],
                 "width": out_image.shape[2],
                 "transform": out_transform})

# Ruta de salida del raster recortado
ruta_salida_i = ruta_raster.replace('.tif', '_gl_CaboLamb.tif')

# Escribir el raster recortado en el disco
with rasterio.open(ruta_salida_i, "w", **out_meta) as dest:
    dest.write(out_image.astype(rasterio.float32))

In [ ]:
# por si paras el procesamiento y seguis otro día, aca queda el codigo de los directorios de los nuevos Rasters creados:
print('# Directorios de los nuevos Rasters creados:')
print(f'ruta_salida_f = "{ruta_salida_f}"')
print(f'ruta_salida_i = "{ruta_salida_i}"')

PERO, tengamos encuenta que los valores NaN de cada DEM (sea por errores, nubes) es diferente, entonces vamos a crear una mascara geográfica de NaN de ambos DEMs para que la diferencia se realice solo en areas con datos

In [ ]:
##############################################################################################
###############  Plotear ambos DEMs para ver la cantidad de pixeles sin datos: ###############
##############################################################################################

# ---------- Abrir _f
with rasterio.open(ruta_salida_f) as src:
    data_f = src.read(1).astype(float)
    nodata = src.nodata
if nodata is not None:
    data_f[data_f == nodata] = np.nan  # Convertir nodata del Raster a NaN
# ---------- Crear array que contenga solo los NaN
array_raster_f_nan = np.where(np.isnan(data_f), np.nan, 1)
print("Cantidad de píxeles de _f sin datos: ", np.isnan(array_raster_f_nan).sum()) # el numero dará muy alto por las afueras del vector_limite_del_glaciar hasta armar un matriz rectangular
# ---------- Abrir _i
with rasterio.open(ruta_salida_i) as src:
    data_i = src.read(1).astype(float)
    nodata = src.nodata
if nodata is not None:
    data_i[data_i == nodata] = np.nan  # Convertir nodata del Raster a NaN
# ---------- Crear array que contenga solo los NaN
array_raster_i_nan = np.where(np.isnan(data_i), np.nan, 1)
print("Cantidad de píxeles de _i sin datos: ", np.isnan(array_raster_i_nan).sum())
print('Si el numero da muy alto es porque el vector_limite_del_glaciar no es rectangular.')

print(' ')
print('Ploteos:')
masked = np.ma.masked_invalid(array_raster_f_nan)
cmap = cm.get_cmap()           # toma el colormap por defecto
cmap.set_bad(color='black')    # los NaN se verán negros
plt.figure(figsize=(8,6))
plt.imshow(masked, cmap=cmap)
plt.title("array_raster_nan (1 = datos, NaN = negro)")
plt.colorbar(label="Valor")
plt.show()
masked = np.ma.masked_invalid(array_raster_i_nan)
cmap = cm.get_cmap()           # toma el colormap por defecto
cmap.set_bad(color='black')    # los NaN se verán negros
plt.figure(figsize=(8,6))
plt.imshow(masked, cmap=cmap)
plt.title("array_raster_nan (1 = datos, NaN = negro)")
plt.colorbar(label="Valor")
plt.show()
print('(El color Negro/Blanco/Transparente corresponde a valores Nan)')

### ENMASCAR LOS DEMs POR VALORES NAN

In [ ]:
# Abrir ambos Rasters
with rasterio.open(ruta_salida_f) as src_f, rasterio.open(ruta_salida_i) as src_i:
    data_f = src_f.read(1).astype(float)
    data_i = src_i.read(1).astype(float)
    nodata_f = src_f.nodata
    nodata_i = src_i.nodata
    if nodata_f is not None:
        data_f[data_f == nodata_f] = np.nan # nodata a NaN
    if nodata_i is not None:
        data_i[data_i == nodata_i] = np.nan # nodata a NaN
    
    # Crear las máscaras
    array_raster_f_nan = np.where(np.isnan(data_f), np.nan, 1)   # Arrays de NaN donde f no tiene datos y el resto REDEFINO A == 1
    array_raster_i_nan = np.where(np.isnan(data_i), np.nan, 1)   # Arrays de NaN donde i no tiene datos y el resto REDEFINO A == 1
    
    # Aplicar las máscaras multiplicando un Raster por el otro
    data_i_masked = data_i * array_raster_f_nan
    data_f_masked = data_f * array_raster_i_nan
    
    # Exportar
    meta_f = src_f.meta.copy() # Utilizamos los meta-datos del raster fuente
    meta_i = src_i.meta.copy() # Utilizamos los meta-datos del raster fuente
meta_i.update(dtype="float32", nodata=np.nan)
ruta_salida_f_2 = ruta_salida_f.replace('.tif', '_masked.tif') # Defino Directorio
ruta_salida_i_2 = ruta_salida_i.replace('.tif', '_masked.tif') # Defino Directorio
with rasterio.open(ruta_salida_i_2, "w", **meta_i) as dst:
    dst.write(data_i_masked.astype("float32"), 1)
meta_f.update(dtype="float32", nodata=np.nan)
with rasterio.open(ruta_salida_f_2, "w", **meta_f) as dst:
    dst.write(data_f_masked.astype("float32"), 1)

In [ ]:
Si me da error de alineación podes usar el siguiente código, sino saltealo!:

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
def reproject_match(src_path, match_path, dst_path):
    with rasterio.open(match_path) as m:           # ráster de referencia
        dst_crs   = m.crs
        dst_aff   = m.transform
        dst_shape = (m.height, m.width)

    with rasterio.open(src_path) as s:
        kwargs = s.meta.copy()
        kwargs.update({
            "crs": dst_crs,
            "transform": dst_aff,
            "height": dst_shape[0],
            "width":  dst_shape[1]
        })

        with rasterio.open(dst_path, "w", **kwargs) as dst:
            for i in range(1, s.count + 1):
                reproject(
                    source=rasterio.band(s, i),
                    destination=rasterio.band(dst, i),
                    src_transform=s.transform,
                    src_crs=s.crs,
                    dst_transform=dst_aff,
                    dst_crs=dst_crs,
                    resampling=Resampling.nearest  # o bilinear/cubic según el caso
                )
                
# 1) Generar Raster alineado al otro Raster 
ruta_f_aligned = ruta_salida_f.replace('.tif', '_aligned-i.tif') # (si tenes espacio limitado sobreescribi)
reproject_match(ruta_salida_f, ruta_salida_i, ruta_f_aligned)

# 2) Vuelve a abrir ambos ráster (el original de referencia y el otro pero alineado)
with rasterio.open(ruta_salida_i) as src_i, rasterio.open(ruta_f_aligned) as src_f:
    data_f = src_f.read(1).astype(float)
    data_i = src_i.read(1).astype(float)
    nodata_f = src_f.nodata
    nodata_i = src_i.nodata
    if nodata_f is not None:
        data_f[data_f == nodata_f] = np.nan # nodata a NaN
    if nodata_i is not None:
        data_i[data_i == nodata_i] = np.nan # nodata a NaN
    
    # Crear las máscaras
    array_raster_f_nan = np.where(np.isnan(data_f), np.nan, 1)   # Arrays de NaN donde f no tiene datos y el resto REDEFINO A == 1
    array_raster_i_nan = np.where(np.isnan(data_i), np.nan, 1)   # Arrays de NaN donde i no tiene datos y el resto REDEFINO A == 1
    
    # Aplicar las máscaras multiplicando un Raster por el otro
    data_i_masked = data_i * array_raster_f_nan
    data_f_masked = data_f * array_raster_i_nan
    
    # Exportar)
    meta_f = src_f.meta.copy() # (Utilizamos los meta-datos del raster fuente)
    meta_i = src_i.meta.copy() # (Utilizamos los meta-datos del raster fuente)
meta_i.update(dtype="float32", nodata=np.nan)
ruta_salida_f_2 = ruta_salida_f.replace('.tif', '_masked.tif') # Defino Directorio
ruta_salida_i_2 = ruta_salida_i.replace('.tif', '_masked.tif') # Defino Directorio
with rasterio.open(ruta_salida_i_2, "w", **meta_i) as dst:
    dst.write(data_i_masked.astype("float32"), 1)
meta_f.update(dtype="float32", nodata=np.nan)
with rasterio.open(ruta_salida_f_2, "w", **meta_f) as dst:
    dst.write(data_f_masked.astype("float32"), 1)

In [ ]:
###################################################################################
##################  Plotear mbos DEM enmascarados para chequear: ##################
###################################################################################

def plot_raster(path, titulo): # Función
    with rasterio.open(path) as src:
        data = src.read(1).astype(float) # 1. leer banda 1
        # 2. enmascarar NaN para que sean transparentes
        masked = np.ma.masked_invalid(data)     
        vmin, vmax = np.nanmin(data), np.nanmax(data)

    plt.figure(figsize=(12,6))
    plt.imshow(masked, vmin=vmin, vmax=vmax, cmap='viridis')
    plt.colorbar(label='Valor')
    plt.title(titulo)
    plt.show()
    
plot_raster(ruta_salida_i_2, "Raster i_2 (DEM inicial enmascarado)")
plot_raster(ruta_salida_f_2, "Raster f_2 (DEM final enmascarado)")

In [ ]:
# por si paras el procesamiento y seguis otro día, aca queda el codigo de los directorios de los nuevos Rasters creados:
print('# Directorios de los nuevos Rasters creados:')
print(f'ruta_salida_f_2 = "{ruta_salida_f_2}"')
print(f'ruta_salida_i_2 = "{ruta_salida_i_2}"')

In [ ]:
# Pegar rutas de arriba (: si necesitas seguir otro día


Ahora si, a continuar!

# DIFERENCIA entre ambos DEMs
enmascarados

In [ ]:
# DEM final
with rasterio.open(ruta_salida_f_2) as src:
    raster_f = src.read(1)
    meta = src.meta
# DEM inicial
with rasterio.open(ruta_salida_i_2) as src:
    raster_i = src.read(1)
assert raster_f.shape == raster_i.shape, "Asegurarse de que los Rasters tienen el mismo tamaño"
# Calcular la diferencia
diferencia = raster_f - raster_i

# EXPORTAR diferencia!!!
with rasterio.open(ruta_salida_dif, 'w', **meta) as dst:
    dst.write(diferencia, 1)  
# Print
#diferencia

In [ ]:
# Primero definamos el percentil:
percentil = 0.1 # MODIFICAR el % del interes
low_p, high_p = obtener_percentiles(diferencia, p=percentil)

# -------------- Ver Diferencia (exportada) --------------
dif = rasterio.open(ruta_salida_dif)
diferencia = dif.read()
median = np.nanmean(diferencia)
print('Promedio de los pixeles de la diferencia: ', median)
plt.figure(figsize = (12,6))
plt.imshow(diferencia[0], vmin = percentil_10, vmax = percentil_9990, cmap = 'viridis') # MODIFICAR los percentiles_ (*)
plt.title("Diferencia")
plt.show()
print(dif.shape)
print(dif.crs)

# -------------- Plotear Histograma (puedes # MODIFICAR los percentiles en: "percentil ="):
histo_dif = plote_porcent(diferencia, banda = f'Diferencia entre DEMs {percentil}%', p = percentil, nodata = None, figsize = (12,6)) #La función limita el ploteo de datos al 0,1%

# -------------- Ver los valores de los datos segun el percentil, # MODIFICAR la x en percentil_X si modificas el "percentil = "
print("Percentil 0.1 %:",  percentil_10) # para elegir el _X de la variable, poner el percentil que usaste por *100
print("Percentil 0.99 %:", percentil_9990) # para elegir el _X de la variable, poner el percentil que usaste por *100
# (*) en mi ej. uso percentil 0.1, las variables seran percentil_(0.1*100) percentil_(100-0.1*100)

# OJO que para el balance de masa tambien llamo a los "percentil_10" y "percentil_9990"

# Vamos a trabajar para el Balance de Masa
dentro de los rangos de percentiles 0.1

In [ ]:
# --------------  Abrir y ver propiedades y datos del Raster Diferencias  --------------
with rasterio.open(ruta_salida_dif) as dif_src:
    dif = dif_src.read(1).astype(float) # Leer la primer banda y convertir a float
    nodata_val = dif_src.nodata # Obtener el valor nodata del archivo
    if nodata_val is not None:
        dif[dif == nodata_val] = np.nan  # Reemplazar nodata por NaN
    media = np.nanmean(dif)
    resolucion = src.res
    res_x, res_y = src.res # [ancho, alto] de cada píxel (debe se _x = a _y)
    valor_px = res_x
    x_filas, y_columnas = dif.shape

### Filtrar
por valores esperados en tu trabajo o percentil, para eso plotiemos diferentes Histogramas con diferentes p=

In [ ]:
# Plotear diferentes histogramas para poder elegir el rango que mas se adapte a tus datos
histo_dif = plote_porcent(dif, banda = f'Diferencia entre DEMs {percentil}%', p = 0.1, nodata = None, figsize = (12,6)) #La función limita el ploteo de datos al 0,1%

In [ ]:
# Vamos a filtrar los datos según el percentil más "lógico"
low_p, high_p = obtener_percentiles(diferencia, p=0.1) # MODIFICAR percentil p=
mascara = (dif > percentil_10) & (dif < percentil_9990) # MODIFICAR "percentil_X" usando X= p*100 y X= (100-p)*100
diferencia = dif[mascara]

In [ ]:
print('Resolución espacial del DEM dif: ', valor_px, '. Es X = Y: ', res_x, '=', res_y, '?')
print('Promedios de los pixeles de la dif: ', media)
print('######################################################################')
total = x_filas * y_columnas
print(f'Cantidad total de píxeles en en Raster como Matriz rectangular: {total }')
cantidad_nan = np.isnan(diferencia).sum()
print(f'Cantidad de píxeles NaN: {cantidad_nan}')
datos_validos = np.sum(~np.isnan(diferencia))
print("Cantidad de píxeles con datos válidos:", datos_validos)
area_datos_validos = (datos_validos * valor_px)/1000000
print('Area del glaciar = ', area_datos_validos, 'km2 (excluyendo NaNs)')

In [ ]:
# -------------  Vector limite del glaciar  --------------
gdf = gpd.read_file(archivo_gl) # esta en EPSG 32721, que ya es coord planas UTM
gdf["area_m2"] = gdf.geometry.area
area_gl = gdf["area_m2"].sum() /1000000
print('Area total del glaciar = ', area_gl, 'km2 (del vector_limite_del_glaciar)')

In [ ]:
# --------------  DETERMINAR AREAS  --------------
# Reasignar valores fuera del rango esperado a CERO
diferencia_ = diferencia.copy()
diferencia_[diferencia_ <= percentil_10] = 0 # MODIFICAR si usaste otro percentil
diferencia_[diferencia_ >= percentil_9990] = 0 # MODIFICAR si usaste otro percentil
# Seleccionar solo datos = a CERO
diferencia_0 = diferencia_[diferencia_ >= 0]
diferencia_0_ = diferencia_0[diferencia_0 <= 0]
datos_errados = len(diferencia_0_)
area_datos_errados = (datos_errados * valor_px)/1000000
print('Area de datos fuera de lo esperado: ', area_datos_errados, 'km2.')

print('Area total del glaciar = ', area_datos_validos, 'km2 (excluyendo NaNs)')
area_sin_datos_errados = area_datos_validos - area_datos_errados
print('Area con datos dentro de lo esperado: ',area_sin_datos_errados, 'km2.')
print('Area total del glaciar = ', area_gl, 'km2 (del vector_limite_del_glaciar)')

# DETERMINAR BALANCE DE MASA
usando valores esperados para un balance de masa (percentiles 0.1)

In [ ]:
# Obtener el valor promedio del Raster diferencia dentro de los valores esperados para un balance de masa (usando los percentiles 0.1)
print('Dentro de los valores esperados para un balance de masa (usando los percentiles 0.1):')
diferencia_ = diferencia.copy()
diferencia_ = diferencia_[diferencia_ > percentil_10] # MODIFICAR si usaste otro percentil
diferencia_ = diferencia_[diferencia_ < percentil_9990] # MODIFICAR si usaste otro percentil
promedio = np.mean(diferencia_)
print('Promedio de la diferencia : ', promedio)
# Obtener la media Positiva y Negativa del Raster diferencia, dentro de los valores esperados
dif_M_0 = diferencia_[diferencia_ >= 0]
dif_m_0 = diferencia_[diferencia_ <= 0]
prom_M = dif_M_0.mean()
prom_m = dif_m_0.mean()
print('Promedio de px POSITIVOS : ', prom_M)
print('Promedio de px NEGATIVOS : ', prom_m)

# Obtener areas y volumenes de Ablacion y Acumulación
len_M = len(dif_M_0)
len_m = len(dif_m_0)
#len_ELA = len(dif_0)

area_ganancia_m = (len_M * valor_px)
area_perdida_m = (len_m * valor_px)
#area_ELA_km2 = (len_ELA * valor_px)

prom_M = dif_M_0.mean()
prom_m = dif_m_0.mean()

vol_ganancia = (area_ganancia_m * prom_M)
vol_perdida = (area_perdida_m * prom_m)

#print(area_ELA_km2)
print('Sup. de ABLACIÓN: ',area_perdida_m/1000000, 'km2.')
print('Sup. de ACUMULACIÓN: ', area_ganancia_m/1000000, 'km2.')
area_perdidaMASganada_m = area_perdida_m + area_ganancia_m
area_perdidaMASganada = (area_perdida_m + area_ganancia_m)/1000000 #(para pasar a km2 porqeu los resultados estan en m)
print(f'area_perdidaMASganada = {area_perdidaMASganada} km2.', f', tiene que dar igual a area_sin_datos_errados: {area_sin_datos_errados} km2.')

ablacion_metros = (vol_perdida / area_perdidaMASganada_m) # no igual a prom_M porque tiene en cuenta toda el area gl
acumulacion_metros = (vol_ganancia / area_perdidaMASganada_m) # no igual a prom_M porque tiene en cuenta toda el area gl
print('Ablación en metros: ', ablacion_metros)
print('Acumulación en metros: ', acumulacion_metros)

# --------------  DETERMINAR EL BALANCE DE MASA  --------------
balance_metros = acumulacion_metros + ablacion_metros
print('Balance total en metros: ', balance_metros)
balance_masa = balance_metros * 0.85 # (Cita: Bader, 1954 y Sapiano et al., 1998)
print('RESULTADO FINAL:')
print(f"Balance de masa: {balance_masa:.3f} m w.e.")

# Podes plotear los Histogramas:
#histo_M = plote_porcent(dif_M_0, banda = 'Pixeles POSITIVOS', p = 0, nodata = None, figsize = (12,6)) # Si queres ver Histograma de valores pos.
#histo_m = plote_porcent(dif_m_0, banda = 'Pixeles NEGATIVOS', p = 0, nodata = None, figsize = (12,6)) # Si queres ver Histograma de valores neg.

### Plotear diferencia enmascarando

In [ ]:
# -------------- Plotear Histograma (puedes # MODIFICAR los percentiles en: "percentil ="):
percentil = 0.1 # MODIFICAR el % del interes
histo_dif = plote_porcent(diferencia, banda = f'Diferencia entre DEMs con p={percentil}%', p = percentil, nodata = None, figsize = (12,6)) #limitando datos hasta percentil= 0.01
low_p, high_p = obtener_percentiles(diferencia, p=0.5) # con 0.5, el 1% del total de los datos queda afuera, que son las colas
print(f'Para ver en visor GIS, definir MIN y MAX en: min={percentil_50}, max={percentil_9950} ; al p=0.5')

In [ ]:
with rasterio.open(ruta_salida_dif) as src:
    data = src.read(1) #banda 1
plt.figure(figsize=(10, 10))
plt.imshow(data, cmap='plasma_r', vmin=percentil_50, vmax=percentil_9950, interpolation='none')
plt.colorbar(label='Valor de píxel')
plt.title('Píxeles entre -0.1 y 0.1 en Violeta')
plt.show()